Las Vegas reports its casino results to the state every month, down to the game. I wanted to know what actually drives that money, and whether the city's recovery since 2020 looks the way the headlines suggest.

> **The question.** What drives Las Vegas Strip casino revenue, and how predictable is it?

The data comes from two public sources, joined month by month:

| Source | What it has | Coverage |
|---|---|---|
| **Nevada Gaming Control Board** | casino win by area and game: slots by denomination, blackjack, baccarat, craps and more | Jan 2010 – Sep 2026 |
| **LVCVA** (Las Vegas Convention and Visitors Authority) | visitors, convention attendance, hotel occupancy, room rates | Jan 2019 – Sep 2026 |

"Win" is what the casinos keep after paying out winning bets: gaming revenue before any expenses. The board publishes it as PDFs, so the first job was a parser that turns 198 monthly reports into a table and checks that every month adds up to the report's own totals. Statewide, Strip and downtown figures pass every month. The model and every metric below are written in SQL (DuckDB); this page queries it directly.

In [1]:
#| collapse
import duckdb
import numpy as np
import pandas as pd
import altair as alt

con = duckdb.connect("../data/vegas.duckdb", read_only=True)
q = lambda sql: con.sql(sql).df()          # run SQL, get a DataFrame

COVID = pd.DataFrame({"start": [pd.Timestamp("2020-03-01")], "end": [pd.Timestamp("2021-06-01")]})
BLUE, ORANGE, AQUA, MUTED = "#3987e5", "#d95926", "#199e70", "#747a87"

q("SELECT count(*) AS area_months, min(month) AS first, max(month) AS latest FROM v_area_monthly WHERE area_key = 'strip'")

,area_months,first,latest
0,201,2010-01-01,2026-09-01


## Sixteen years on the Strip

Monthly revenue is noisy, so I plot it alongside a rolling 12-month total, which smooths out seasons and lucky months and shows the underlying trend.

In [2]:
strip = q('''
    SELECT month, total_win / 1e6 AS win_m, win_12m / 1e9 AS win_12m_bn, yoy_12m
    FROM v_area_monthly WHERE area_key = 'strip' ORDER BY month
''')
strip.tail(3)

,month,win_m,win_12m_bn,yoy_12m
198,2026-07-01,776.339,8.980370,0.018127
199,2026-08-01,684.087,8.985096,0.014557
200,2026-09-01,693.467,8.990752,0.019784


In [3]:
#| hide
#| caption: Monthly Strip win (bars) and the average of the last 12 months (line). The shaded band is the COVID period. Hover for the 12-month total.
shade = alt.Chart(COVID).mark_rect(color="#747a87", opacity=0.12).encode(x="start:T", x2="end:T")
bars = alt.Chart(strip).mark_bar(color=BLUE, opacity=0.55, width=2).encode(
    x=alt.X("month:T", title=None),
    y=alt.Y("win_m:Q", title="Monthly win ($M)"),
    tooltip=[alt.Tooltip("month:T", format="%b %Y", title="Month"),
             alt.Tooltip("win_m:Q", format=",.0f", title="Win ($M)"),
             alt.Tooltip("win_12m_bn:Q", format=".2f", title="Last 12 months ($B)")],
)
avg = alt.Chart(strip.dropna(subset=["win_12m_bn"])).transform_calculate(
    monthly_avg="datum.win_12m_bn * 1000 / 12").mark_line(color="#e8e9ec", strokeWidth=2).encode(
    x="month:T", y="monthly_avg:Q")
(shade + bars + avg).properties(title="Strip win per month, with the 12-month average", height=300)

alt.LayerChart(...)

The Strip earned about **\$6.6 billion a year before COVID** and has settled at roughly **\$8.8 billion** since 2023. That's a step up of a third, not a gradual climb: the 12-month line jumps after reopening, then flattens. 2025 and 2026 are essentially level with 2023, so the post-reopening surge is over and the Strip has found a new plateau.

## Fewer visitors, more money

The obvious explanation for higher revenue is more visitors. The data says the opposite. To compare two measures on very different scales without a misleading second axis, I index both to 2019 = 100.

In [4]:
indexed = q('''
    WITH yearly AS (
        SELECT year(month) AS year, sum(strip_win) AS win, sum(visitors) AS visitors, count(*) AS n
        FROM v_strip_tourism GROUP BY ALL HAVING count(*) = 12
    )
    SELECT year,
           100 * win / first(win) OVER (ORDER BY year)           AS "Strip win",
           100 * visitors / first(visitors) OVER (ORDER BY year) AS "Visitors",
           win / visitors                                        AS win_per_visitor
    FROM yearly ORDER BY year
''')
indexed.round(1)

,year,Strip win,Visitors,win_per_visitor
0,2019,100.0,100.0,154.9
1,2020,56.7,44.8,196.2
2,2021,107.6,75.8,219.9
3,2022,125.8,91.3,213.4
4,2023,135.2,96.0,218.0
5,2024,133.8,98.0,211.5
6,2025,133.8,90.6,228.7


In [5]:
#| hide
#| caption: Annual totals indexed to 2019 = 100. 2020 and 2021 are dominated by the closure.
long = indexed.melt(id_vars="year", value_vars=["Strip win", "Visitors"], var_name="measure", value_name="index")
base = alt.Chart(long).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("index:Q", title="Index (2019 = 100)", scale=alt.Scale(domain=[30, 150])),
    color=alt.Color("measure:N", title=None, scale=alt.Scale(domain=["Strip win", "Visitors"], range=[BLUE, ORANGE])),
    tooltip=["year:O", "measure:N", alt.Tooltip("index:Q", format=".0f")],
)
rule = alt.Chart(pd.DataFrame({"y": [100]})).mark_rule(strokeDash=[4, 4], color=MUTED).encode(y="y:Q")
(rule + base.mark_line(strokeWidth=2) + base.mark_point(size=70, filled=True)).properties(
    title="Revenue is up a third; visitors are still below 2019", height=280)

alt.LayerChart(...)

Visitors haven't recovered to 2019 levels, yet the Strip earns about a third more. **Each visitor is worth more:** Strip win per Las Vegas visitor rose from about \$155 in 2019 to between \$210 and \$230 since 2022. Room rates tell the same story, with the Strip's average daily rate up sharply while occupancy is still below its pre-COVID peak.

The likely reading is a deliberate shift toward higher-spending guests: pricier rooms, more premium gaming, fewer budget trips. The data can't prove the cause, but it rules out the simple "more people came back" story.

## Slots are taking over

The Strip used to be close to an even split between table games and slot machines. It isn't any more.

In [6]:
# Shares of each year's win, using only months with game-level detail
# (three 2025 months have totals only, so raw sums would understate that year).
mix = q('''
    WITH grouped AS (
        SELECT year(month) AS year,
               CASE WHEN game_group IN ('Roulette', 'Poker-style table games', 'Sports & race', 'Poker room', 'Other tables')
                    THEN 'Other table games' ELSE game_group END AS game_group,
               sum(win) AS win
        FROM v_game_mix
        WHERE area_key = 'strip' AND year(month) BETWEEN 2010 AND 2025
        GROUP BY 1, 2
    )
    SELECT year, game_group, win / sum(win) OVER (PARTITION BY year) AS share FROM grouped
''')
share = q('''
    SELECT year(month) AS year, sum(slots_win) / sum(total_win) AS slot_share
    FROM v_area_monthly WHERE area_key = 'strip' AND slots_win IS NOT NULL AND year(month) <= 2025
    GROUP BY ALL ORDER BY year
''')
share[share.year.isin([2010, 2015, 2019, 2025])].round(3)

,year,slot_share
0,2010,0.483
5,2015,0.482
9,2019,0.523
15,2025,0.562


In [7]:
#| hide
#| caption: Each game group's share of annual Strip win. Multi-denomination slots are the modern video machines that let a player choose the bet size.
order = ["Multi-denomination slots", "Penny slots", "High-limit slots ($5+)", "Other slots",
         "Baccarat", "Blackjack", "Craps", "Other table games"]
# slots in blues, table games in warm colours (this site's palette)
colors = ["#2340ff", "#6a7cff", "#a3aeff", "#d3d8ff", "#eb6834", "#eda100", "#1baf7a", "#b9b6ad"]
alt.Chart(mix).mark_area().encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0, values=[2010, 2013, 2016, 2019, 2022, 2025])),
    y=alt.Y("share:Q", title="Share of Strip win", stack="normalize", axis=alt.Axis(format="%")),
    color=alt.Color("game_group:N", title=None, sort=order, scale=alt.Scale(domain=order, range=colors),
                    legend=alt.Legend(columns=2, symbolType="square")),
    order=alt.Order("order_idx:Q"),
    tooltip=["year:O", alt.Tooltip("game_group:N", title="Group"), alt.Tooltip("share:Q", format=".1%", title="Share")],
).transform_calculate(order_idx="indexof(" + str(order) + ", datum.game_group)").properties(
    title="Slots (blues) now earn more than half the Strip's win", height=320)

alt.Chart(...)

Slots went from about **48% of Strip win in 2010 to 56% in 2025**. The growth is in multi-denomination video slots, which went from about \$1.3 billion in 2019 to \$3.4 billion in 2024. Penny slots nearly doubled through 2019 and then fell back, and high-limit slots are a small slice. Part of that swap is probably classification rather than behaviour: modern machines offer several bet sizes, so play that used to be recorded as "penny" now lands in "multi-denomination". Taken together, though, slots clearly grew faster than tables.

This matters for anyone forecasting the Strip: slot revenue is driven by many small bets and is steady. Table revenue, as the next section shows, is not.

## Baccarat: the wild card

Baccarat is played for very high stakes by a small number of players. The house's long-run edge is small, so in any one month the result depends heavily on whether a few big players win or lose. The share of bets the house keeps is called the **hold**.

In [8]:
bac = q("SELECT month, hold, hold_trailing_24m, baccarat_win / 1e6 AS win_m FROM v_baccarat WHERE month >= '2022-01-01' ORDER BY month")
# How much does each line move from one month to the next? (median absolute change, 2022 on)
swing = q('''
    WITH g AS (
        SELECT game_group, month, win / LAG(win) OVER (PARTITION BY game_group ORDER BY month) - 1 AS change
        FROM v_game_mix
        WHERE area_key = 'strip' AND month >= '2022-01-01'
          AND game_group IN ('Baccarat', 'Blackjack', 'Multi-denomination slots')
    )
    SELECT game_group AS "Game group",
           median(abs(change))              AS "Typical monthly change",
           quantile_cont(abs(change), 0.9)  AS "1 month in 10 moves at least"
    FROM g GROUP BY ALL ORDER BY 2 DESC
''')
swing.round(2)

,Game group,Typical monthly change,1 month in 10 moves at least
0,Baccarat,0.29,0.97
1,Blackjack,0.17,0.44
2,Multi-denomination slots,0.07,0.16


In [9]:
#| hide
#| caption: Monthly baccarat hold on the Strip (dots) against its trailing two-year average (line). Hover for the month's baccarat win.
pts = alt.Chart(bac).mark_circle(size=55, color=ORANGE, opacity=0.9).encode(
    x=alt.X("month:T", title=None),
    y=alt.Y("hold:Q", title="Hold (share of bets kept)", axis=alt.Axis(format="%")),
    tooltip=[alt.Tooltip("month:T", format="%b %Y", title="Month"), alt.Tooltip("hold:Q", format=".1%", title="Hold"),
             alt.Tooltip("win_m:Q", format=",.0f", title="Baccarat win ($M)")],
)
line = alt.Chart(bac).mark_line(color="#e8e9ec", strokeWidth=2).encode(x="month:T", y="hold_trailing_24m:Q")
(line + pts).properties(title="Baccarat hold swings around its average", height=280)

alt.LayerChart(...)

In a typical month since 2022, baccarat win moves about **29%** from the month before; multi-denomination slots move about **7%**, so baccarat is roughly four times as jumpy. One month in ten, baccarat nearly doubles or halves. A single month can swing by over \$100 million: in September 2026 the house kept under 10% of baccarat bets against a two-year average near 16%, and baccarat win fell to \$59 million from \$157 million the month before.

So when a monthly headline says "Strip revenue fell", the first question should be **what did baccarat do?** It's often luck, not demand.

## The calendar

Each month's revenue compared with its year's monthly average, across 14 normal years (2010–2019 and 2022–2025, excluding COVID and the partial current year).

In [10]:
season = q("SELECT month_num, month_name, index, index_min, index_max FROM v_seasonality WHERE area_key = 'strip' ORDER BY month_num")
season[["month_name", "index"]].set_index("month_name").T.round(2)

month_name,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
index,1.01,1.05,0.98,0.88,1.01,0.96,1.03,0.97,1.0,1.02,0.98,1.12


In [11]:
#| hide
#| caption: Average seasonal index (dots) with the range across 14 years (lines). 1.0 is an average month.
base = alt.Chart(season).encode(x=alt.X("month_name:N", sort=list(season.month_name), title=None, axis=alt.Axis(labelAngle=0)))
rng = base.mark_rule(color=MUTED, strokeWidth=2).encode(y=alt.Y("index_min:Q", title="Index (1 = average month)", scale=alt.Scale(domain=[0.75, 1.3])), y2="index_max:Q")
dot = base.mark_point(size=90, filled=True, color=BLUE).encode(
    y="index:Q", tooltip=[alt.Tooltip("month_name:N", title="Month"), alt.Tooltip("index:Q", format=".2f", title="Average"),
                          alt.Tooltip("index_min:Q", format=".2f", title="Lowest"), alt.Tooltip("index_max:Q", format=".2f", title="Highest")])
one = alt.Chart(pd.DataFrame({"y": [1]})).mark_rule(strokeDash=[4, 4], color=MUTED).encode(y="y:Q")
(one + rng + dot).properties(title="December peaks, April dips", height=260)

alt.LayerChart(...)

**December is the strongest month**, about 12% above average (holiday travel, New Year's Eve, and big baccarat play around Lunar New Year can fall in Dec–Feb), and **April is the weakest**, about 12% below. The ranges are wide, again largely baccarat, so seasonality is a tendency, not a rule.

## How predictable is it?

To test this honestly, I trained each model on January 2022 – September 2025 (after the reopening rebound) and scored it on the following twelve months, which it never saw. The score is the average miss as a percentage of the actual figure (MAPE).

- **Seasonal naive:** next October = last October. The baseline any model has to beat.
- **Naive with growth:** the same, scaled by the last year's growth.
- **Holt-Winters:** exponential smoothing with a damped trend and a seasonal pattern.

In [12]:
#| collapse
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")

s = q("SELECT month, total_win, slots_win FROM v_area_monthly WHERE area_key = 'strip' AND month >= '2022-01-01' ORDER BY month")
s = s.set_index(pd.DatetimeIndex(s.month, freq="MS"))
s["slots_win"] = s["slots_win"].interpolate()   # 3 months in 2025 have totals only

def score(y):
    train, test = y[:"2025-09"], y["2025-10":]
    mape = lambda p: float(np.mean(np.abs(p.values - test.values) / test.values))
    naive = y.shift(12)["2025-10":]
    growth = train[-12:].sum() / train[-24:-12].sum()
    hw = ExponentialSmoothing(train, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit()
    return {"Seasonal naive": mape(naive), "Naive with growth": mape(naive * growth),
            "Holt-Winters": mape(hw.forecast(len(test)))}

scores = pd.DataFrame({"Total win": score(s.total_win), "Slots only": score(s.slots_win)})
scores.map(lambda v: f"{v:.1%}")

,Total win,Slots only
Seasonal naive,5.4%,5.0%
Naive with growth,5.7%,4.7%
Holt-Winters,3.8%,3.1%


In [13]:
#| hide
#| caption: Holt-Winters refit on all data since 2022, forecasting six months ahead. The band is ± twice the typical holdout error, a rough guide rather than a formal interval.
hw_all = ExponentialSmoothing(s.total_win, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit()
fc = hw_all.forecast(6)
err = float(scores.loc["Holt-Winters", "Total win"])
fdf = pd.DataFrame({"month": fc.index, "win_m": fc.values / 1e6})
fdf["lo"], fdf["hi"] = fdf.win_m * (1 - 2 * err), fdf.win_m * (1 + 2 * err)
hist = pd.DataFrame({"month": s.index, "win_m": s.total_win.values / 1e6})
hist = hist[hist.month >= "2024-01-01"]
band = alt.Chart(fdf).mark_area(color=BLUE, opacity=0.18).encode(x="month:T", y="lo:Q", y2="hi:Q")
actual = alt.Chart(hist).mark_line(color="#e8e9ec", strokeWidth=2).encode(
    x=alt.X("month:T", title=None), y=alt.Y("win_m:Q", title="Strip win ($M)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("win_m:Q", format=",.0f", title="Actual ($M)")])
pred = alt.Chart(fdf).mark_line(color=BLUE, strokeWidth=2, strokeDash=[5, 3]).encode(
    x="month:T", y="win_m:Q",
    tooltip=[alt.Tooltip("month:T", format="%b %Y"), alt.Tooltip("win_m:Q", format=",.0f", title="Forecast ($M)"),
             alt.Tooltip("lo:Q", format=",.0f", title="Low"), alt.Tooltip("hi:Q", format=",.0f", title="High")])
(band + actual + pred).properties(title="Next six months: forecast (dashed) after actuals", height=280)

alt.LayerChart(...)

Holt-Winters beats both naive baselines, missing by under 4% a month on total win and about 3% on slots alone. That's useful accuracy for planning, but notice how little better it is than "same as last year": the Strip is a mature, seasonal business, and the remaining error is mostly baccarat luck that no model can see coming. Forecasting slots and tables separately, and treating baccarat as a range rather than a number, would be the honest next step.

## What it adds up to

1. **The Strip found a higher plateau, not a growth curve.** About \$8.8B a year since 2023, level for three years.
2. **Value per visitor, not volume, drove the recovery.** Fewer visitors than 2019, a third more revenue.
3. **Slots are now the base of the business** and the predictable part of it.
4. **Read monthly headlines through baccarat.** A single month's swing is often luck at a few high-stakes tables.
5. **Seasonality is real but loose:** December strong, April weak, wide ranges.

## What this does not show

- Totals hide properties. Statewide reporting can't separate one resort's performance from another's.
- "Win" is revenue before expenses. Free play, comps and promotions sit outside it, so margins aren't visible here.
- Visitor counts cover all of Las Vegas, so win per visitor is an index, not a measure of what an individual spends.
- April and May 2020 come from the LVCVA because the board's reports for those months are scanned images, and three 2025 months have totals but no game detail.

The pipeline, SQL model and data tests are in the [project folder on GitHub](https://github.com/mckeeander/mckeeander.github.io/tree/main/projects/vegas-gaming).

*Sources: Nevada Gaming Control Board, Monthly Gaming Revenue Reports; Las Vegas Convention and Visitors Authority, monthly visitor statistics.*